# 03 — Issue Classification

**Project:** Customer Complaint Root-Cause Classifier  
**Module:** 2 — Supervised Classification of Complaint Issues  
**Target:** `issue` (21 categories)  
**Input:** `complaint_narrative`  

### Objectives:
1. Load and prepare complaints data.
2. Implement **Group-Aware Splitting** (`GroupShuffleSplit` on narrative hash) to prevent data leakage from exact-duplicate narratives (identified in EDA).
3. Extract sparse n-gram features via **TF-IDF Vectorizer** (optimized for 8 GB RAM).
4. Train and compare 3 baseline algorithms:
   - **TF-IDF + Multinomial Logistic Regression** (balanced class weights)
   - **TF-IDF + Linear Support Vector Classifier (LinearSVC)**
   - **TF-IDF + Multinomial Naive Bayes**
5. Evaluate using Macro F1, Weighted F1, Precision, Recall, and Confusion Matrices.
6. Save the champion model, vectorizer, and label encoder to `models/` for downstream dashboard and RAG usage.

## 1. Setup and Environment

In [ ]:
import sys
import os
from pathlib import Path

# Ensure project root is in sys.path
project_root = Path(os.path.abspath("")).resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

from config.settings import (
    PROJECT_ROOT, MODELS_DIR, RANDOM_SEED, TEST_SIZE, TFIDF_MAX_FEATURES
)
from config.labels import ISSUE_LABELS
from src.data.load_data import load_complaints
from src.data.clean_data import clean_complaints
from src.utils.helpers import generate_narrative_hash, get_memory_usage
from src.classification.evaluate import evaluate_model, plot_confusion_matrix, compare_models

print(f"Project root: {PROJECT_ROOT}")
print(f"Current memory usage: {get_memory_usage():.2f} MB")

## 2. Load and Inspect Data

In [ ]:
print("Loading and cleaning complaint records...")
raw_df = load_complaints()
df = clean_complaints(raw_df)

print(f"Loaded {len(df):,} cleaned complaints.")
print(f"Unique issues: {df['issue'].nunique()}")
print(f"Target class distribution:\n{df['issue'].value_counts()}")

## 3. Group-Aware Train / Test Split

> **Methodological safeguard**: EDA identified **28,799 duplicate narratives** in the synthetic dataset. Using a naive random split would place identical narratives in both training and test sets, causing artificial data leakage. We use `GroupShuffleSplit` grouping by the MD5 hash of each narrative so identical texts strictly reside in the same partition.

In [ ]:
X = df["complaint_narrative"]
y_raw = df["issue"]

# Generate MD5 group hashes for narratives
print("Generating narrative group hashes...")
groups = X.apply(generate_narrative_hash)

# Encode categorical target labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)
classes = label_encoder.classes_

print(f"Total unique groups (unique narratives): {groups.nunique():,}")

# Group-aware split
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"Train size: {len(X_train):,} ({len(X_train)/len(df):.1%})")
print(f"Test size:  {len(X_test):,} ({len(X_test)/len(df):.1%})")

# Verify zero group overlap
train_groups = set(groups.iloc[train_idx])
test_groups = set(groups.iloc[test_idx])
overlap = train_groups.intersection(test_groups)
print(f"Group leakage / overlap: {len(overlap)} (Must be 0)")
assert len(overlap) == 0, "Data leakage detected between train and test sets!"

## 4. Sparse Feature Extraction (TF-IDF)

We extract unigram + bigram representations with sublinear TF scaling to bound memory consumption under 8 GB RAM.

In [ ]:
print(f"Fitting TF-IDF Vectorizer (max_features={TFIDF_MAX_FEATURES:,}, n-grams=(1,2))...")
vectorizer = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=3,
    max_df=0.95,
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"X_train shape: {X_train_vec.shape}")
print(f"X_test shape:  {X_test_vec.shape}")
print(f"Sparse matrix memory: {X_train_vec.data.nbytes / 1e6:.1f} MB")
print(f"Current memory usage: {get_memory_usage():.2f} MB")

## 5. Model Training & Comparison

We evaluate three models on the exact same test partition:
1. **Logistic Regression (Multinomial, Balanced Class Weights)**
2. **Linear Support Vector Classifier (LinearSVC)**
3. **Multinomial Naive Bayes (alpha=0.1)**

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        C=1.0,
        solver="lbfgs",
        class_weight="balanced",
        random_state=RANDOM_SEED,
        n_jobs=-1
    ),
    "Linear SVM": LinearSVC(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_SEED,
        dual="auto"
    ),
    "Multinomial Naive Bayes": MultinomialNB(alpha=0.1)
}

evaluation_results = []
trained_models = {}
predictions = {}

for name, clf in models.items():
    print(f"\n--- Training: {name} ---")
    clf.fit(X_train_vec, y_train)
    trained_models[name] = clf
    
    y_pred = clf.predict(X_test_vec)
    predictions[name] = y_pred
    
    metrics = evaluate_model(y_test, y_pred, labels=classes, model_name=name)
    evaluation_results.append(metrics)
    
    print(f"{name} — Accuracy: {metrics['accuracy']:.4f} | Macro F1: {metrics['macro_f1']:.4f} | Weighted F1: {metrics['weighted_f1']:.4f}")

## 6. Performance Comparison

In [ ]:
comparison_df = compare_models(evaluation_results)
print("=== MODEL BENCHMARK COMPARISON ===")
print(comparison_df.to_string(index=False))
comparison_df

In [ ]:
# Visualize Model Metric Comparison
metric_cols = ["Accuracy", "Macro F1", "Weighted F1", "Macro Precision", "Macro Recall"]
plot_df = comparison_df.set_index("Model")[metric_cols]

ax = plot_df.plot(kind="bar", figsize=(12, 6), colormap="viridis", edgecolor="black")
plt.title("Comparison of Issue Classification Models", fontsize=14, fontweight="bold")
plt.ylabel("Score", fontsize=12)
plt.ylim(0.0, 1.05)
plt.xticks(rotation=0)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 7. Detailed Evaluation: Champion Model

In [ ]:
best_model_name = comparison_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
best_metrics = [m for m in evaluation_results if m["model_name"] == best_model_name][0]

print(f"Selected Champion Model: {best_model_name}")
print("=" * 60)
print(best_metrics["classification_report_str"])

In [ ]:
# Confusion Matrix for the Champion Model
fig = plot_confusion_matrix(
    y_test,
    predictions[best_model_name],
    labels=classes,
    model_name=best_model_name,
    figsize=(14, 11)
)
plt.show()

## 8. Save Champion Model & Artifacts

In [ ]:
os.makedirs(MODELS_DIR, exist_ok=True)

model_file = MODELS_DIR / "issue_classifier.joblib"
vec_file = MODELS_DIR / "tfidf_issue_vectorizer.joblib"
le_file = MODELS_DIR / "issue_label_encoder.joblib"

print(f"Saving best model ({best_model_name}) -> {model_file}")
joblib.dump(best_model, model_file)

print(f"Saving TF-IDF vectorizer -> {vec_file}")
joblib.dump(vectorizer, vec_file)

print(f"Saving label encoder -> {le_file}")
joblib.dump(label_encoder, le_file)

print("\nAll Module 2 model artifacts saved successfully!")

## 9. Interactive Test Prediction

In [ ]:
from src.classification.predict import predict_issue

sample_complaints = [
    "I was charged a late fee on my credit card even though I sent payment three days before the deadline.",
    "Someone opened an unauthorized account in my name and my credit score dropped by 80 points.",
    "I requested a payoff quote for my vehicle lease but customer service has refused to provide it.",
    "My online banking password keeps resetting and I cannot access my savings account."
]

print("=== SAMPLE PREDICTIONS ===\n")
for text in sample_complaints:
    result = predict_issue(text)
    conf_str = f"{result['confidence']:.2%}" if result['confidence'] is not None else "N/A (Linear SVM)"
    print(f"Narrative:   {text}")
    print(f"Prediction:  {result['predicted_issue']} (Confidence: {conf_str})")
    print("-" * 70)